# 02 · Tools: plain Python functions

An *agent* is a language model that can call functions. Before we build the agent (notebook 03) we
write the functions it will call. They are ordinary Python.

Our agent needs three tools to summarise the reviews of a product:

| tool | what it does | built with |
|---|---|---|
| `search_reviews` | find web pages that review the product | `ddgs` (DuckDuckGo, no API key) |
| `fetch_page` | download one page and keep only its readable text | `httpx` + `trafilatura` |
| `analyze_reviews` | turn review texts into a table of aspects, sentiments and quotes | the model (structured output) + `numpy` |

The third tool is the customer-analytics part, and it is built the way production systems build it.
Amazon's *"Customers say"* box on a product page is not a free-form summary: every review is split into
opinions, each opinion is tagged with a product *aspect* (sound quality, comfort, price, …) and a sentiment,
the tags are **counted**, and only then does a model write a paragraph grounded in those counts and in
quotes. We build a small version of that pipeline.

Note that this tool calls the model itself, with a fixed prompt and a fixed output format. That is
different from the agent in notebook 03, which decides *what* to call: a tool does one fixed job.
Production pipelines run this as a batch over thousands of reviews; the agent is our teaching vehicle.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
MODEL = os.environ["LLM_MODEL"]
EMBEDDING_MODEL = os.environ.get("EMBEDDING_MODEL", "gemini-embedding-001")

## 1. Search

DuckDuckGo can be queried without an account. Each result has a title, a URL and a short text snippet.
We keep only those three fields: everything a tool returns ends up inside the prompt as tokens, so
tools should return what the model needs and nothing more.

Results change from run to run and are not curated: an advert or an affiliate page can appear among the
reviews. The agent will have to live with that, as it would with any real data source.

In [2]:
from ddgs import DDGS


def search_reviews(query: str, max_results: int = 10) -> list[dict]:
    """Search the web for reviews of a product. Returns title, url and snippet per result."""
    results = DDGS().text(f"{query} reviews", max_results=max_results)
    # Keep only what the agent needs; every field here ends up as tokens in the prompt.
    return [{"title": r["title"], "url": r["href"], "snippet": r["body"]} for r in results]


results = search_reviews("Sony WH-1000XM5")
results

[{'title': 'Sony WH-1000XM5 review - SoundGuys',
  'url': 'https://www.soundguys.com/sony-wh-1000xm5-review-71783/',
  'snippet': 'The Sony WH-1000XM5 earns its spot among a crowded field of active noise canceling (ANC) headphones by offering a wide range of new features.'},
 {'title': 'Sony WH-1000XM5 review - What Hi-Fi?',
  'url': 'https://www.whathifi.com/reviews/sony-wh-1000xm5',
  'snippet': "We originally reviewed the Sony WH-1000XM5 back in May 2022 and although they have now been superseded by the Sony WH-1000XM6, the XM5 remain a highly recommendable pair of wireless headphones. Shop around online and your'e likely to find a discount which makes their engaging sound quality, impressive noise-cancelling, great call quality and comfortable fit a match for any rival at the money."},
 {'title': 'Sony WH-1000XM5 Review - PCMag',
  'url': 'https://www.pcmag.com/reviews/sony-wh-1000xm5',
  'snippet': "Sony's WH-1000XM5 headphones pair top-notch noise cancellation with class-leading 

## 2. Fetch

A snippet is a sentence or two; the actual review is on the page. `httpx` downloads it and
`trafilatura` extracts the main text, dropping menus, ads, cookie banners and scripts.

Two design decisions in this function matter for the agent:

* **Errors are returned, not raised.** Some sites block scripted downloads (you will see a `403` below).
  If the function raised an exception the whole agent would stop. Returning an `ERROR: …` string lets the
  model read what happened and decide what to do, e.g. try the next page.
* **The text is capped.** A review page is typically 20 000 to 25 000 characters, about 5 000 tokens, and
  the useful part is the whole article: the first couple of thousand characters are usually disclaimers,
  score tables and update logs. So the cap has to be generous. `max_chars=20000` keeps almost every review
  intact and only protects us from pathological pages. The arithmetic matters because the agent resends
  the whole conversation on every loop iteration: three pages ≈ 15 000 tokens, times four or five
  iterations, is 60 000 to 75 000 tokens per run. That is fine on Gemini's free tier. On Groq, whose free
  tier allows 8 000 tokens per minute, lower the cap to about 2 000 characters.

In [3]:
import httpx
import tiktoken
import trafilatura


def fetch_page(url: str, max_chars: int = 20000) -> str:
    """Download a web page and return its readable text, cut to max_chars."""
    try:
        r = httpx.get(url, follow_redirects=True, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()  # turns 403, 404, 500 ... into an exception
    except httpx.HTTPError as e:
        # A tool must never crash the agent: report the problem as text instead.
        return f"ERROR: could not fetch {url}: {e}"
    text = trafilatura.extract(r.text) or ""  # strips menus, ads, scripts; keeps the article
    return text[:max_chars]


# First without a cap, to see how big a page really is (in characters and in tokens).
enc = tiktoken.get_encoding("o200k_base")
full = fetch_page(results[0]["url"], max_chars=None)
print(f"full page: {len(full)} chars, {len(enc.encode(full))} tokens\n")

# Then with the default, as the agent will call it.
pages = [fetch_page(r["url"]) for r in results[:4]]
for p in pages:
    print(len(p), "chars |", p[:120].replace("\n", " "), "...\n")

texts = [p for p in pages if not p.startswith("ERROR")]  # keep the pages we actually got

full page: 25827 chars, 5960 tokens



20000 chars | All products featured are independently chosen by us. However, SoundGuys may receive a commission on orders placed throu ...

20000 chars | What Hi-Fi? Verdict It can be tricky for a manufacturer to push the sound performance of a product consistently from gen ...

243 chars | ERROR: could not fetch https://www.pcmag.com/reviews/sony-wh-1000xm5: Client error '403 Forbidden' for url 'https://www. ...

20000 chars | In this article, I’m going to give my honest Sony WH-1000XM5 Review with comparisons vs XM6, XM4, Bose QuietComfort Ultr ...



In [4]:
# The start of one page: verdict first, then the review proper.
print(texts[0][:3000], "\n[...]")

All products featured are independently chosen by us. However, SoundGuys may receive a commission on orders placed through its retail links. See our ethics statement.
Sony WH-1000XM5 review
Sep 11, 2025 — 9:46 AM ET
Measured in our lab
See all lab data →All lab data →
The bottom line
The predecessors to the Sony WH-1000XM5 were king of the hill, and this new headphone takes that crown. With new drivers, features, and design, the Sony WH-1000XM5 is an intriguing set of headphones among the flagship active noise canceling (ANC) devices on the market. It excels in travel or at the office, in particular.
What we like
What we don't like
Our assessment
Editor-scored from hands-on use, not measured
- Isolation / Attenuation
- 9.8
- Durability / Build Quality
- 6.7
- Design
- 7.3
- Connectivity
- 9.0
- Portability
- 7.5
- Comfort
- 8.8
- Features
- 9.2
- Value
- 8.4
- Microphone
- 7.5
- Product Release date
- May 20, 2022
- Original price
- $399
- Dimensions (L x W x H mm)
- 225 x 77 x 264 mm


## 3. Analyse: from review text to a "Customers say" table

The pipeline has three steps, and the split between them is the whole point:

1. **Extract** (the model): read a review and list every opinion as *(aspect, sentiment, quote)*.
   Reading prose and naming what it is about is what language models are good at.
2. **Merge** (embeddings): different reviews name the same aspect differently; map near-duplicates
   onto one name.
3. **Aggregate** (plain Python): count opinions per aspect and sentiment, keep a couple of quotes each.
   Counting is what models are *bad* at and code is perfect at.

The result is a table, not a paragraph. In notebook 03 the agent will write the paragraph from the table.

### 3a. Extract opinions with structured output

We ask the model for JSON, and we do not just hope it complies: `response_format` with a JSON schema makes
the server constrain the output to that schema. The `enum` limits sentiment to three values, and the
`quote` field must be the exact words from the text, so every extracted opinion can be checked against
its source.

The last sentence of the prompt matters more than it looks: without "be exhaustive", the model skims a
long review and returns five or six opinions; with it, two to three times as many. Prompts are part of the
tool and deserve the same testing as code. Try removing that sentence and re-running the cell.

In [5]:
import json

import pandas as pd

OPINION_SCHEMA = {
    "type": "object",
    "properties": {
        "opinions": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "aspect": {"type": "string"},
                    "sentiment": {"type": "string", "enum": ["positive", "negative", "mixed"]},
                    "quote": {"type": "string"},
                },
                "required": ["aspect", "sentiment", "quote"],
            },
        }
    },
    "required": ["opinions"],
}

EXTRACT_PROMPT = (
    "You extract opinions about a product from a review. For every opinion return the product aspect it "
    "is about (a short generic name such as 'noise cancelling', 'battery life', 'price', 'comfort'), the "
    "sentiment, and the exact words from the text that express it. Ignore text that is not an opinion "
    "about the product. Be exhaustive: go through the whole text section by section and list every "
    "opinion you find; a long review typically contains 15 to 40 opinions."
)


def extract_opinions(text: str) -> list[dict]:
    """Ask the model to list every opinion in a review as (aspect, sentiment, quote)."""
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": EXTRACT_PROMPT}, {"role": "user", "content": text}],
        response_format={"type": "json_schema", "json_schema": {"name": "opinions", "schema": OPINION_SCHEMA}},
    )
    return json.loads(response.choices[0].message.content)["opinions"]


opinions = extract_opinions(texts[0])
print(len(opinions), "opinions found in the first review")
pd.DataFrame(opinions).head(12)

17 opinions found in the first review


,aspect,sentiment,quote
0,design,positive,"With new drivers, features, and design, the So..."
1,noise cancelling,positive,this is one of the best ANC headphones on the ...
2,price,negative,even at an eye-watering $400.
3,features,positive,"The WH-1000XM5 has many bells and whistles, an..."
4,comfort,positive,the padding of the Sony WH-1000XM5 is deep and...
5,weight,positive,the WH-1000XM5 is light enough to spare your n...
6,ear sensor,positive,The ear sensor is quite good at pausing music ...
7,moisture resistance,negative,these may not be the headphones for you. The h...
8,app,negative,"there are also a few pitfalls. Namely, it asks..."
9,battery life,positive,Our standard battery test yielded a result of ...


### 3b. Merge aspect names with embeddings

One review says *noise cancelling*, another *noise cancellation*. To count them together we need to know
that the two names mean the same thing, and string matching will not tell us.

An **embedding** is a vector that represents the meaning of a piece of text: texts with similar meaning
get vectors pointing in similar directions. The *cosine similarity* between two vectors (their dot product
after normalising to length 1) is close to 1 for near-synonyms and lower for unrelated terms. The
embedding model is called through the same client as the chat model.

The merge is greedy: walk through the aspect names from most to least frequent; if a name is similar
enough to one we already kept, map it onto that one, otherwise keep it as a new aspect.

In [6]:
from collections import Counter

import numpy as np


def embed(texts: list[str]) -> np.ndarray:
    """One unit-length vector per text, as rows of a matrix."""
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    vectors = np.array([d.embedding for d in response.data])
    return vectors / np.linalg.norm(vectors, axis=1, keepdims=True)


def merge_aspects(names: list[str], threshold: float = 0.8) -> dict[str, str]:
    """Map each aspect name to a canonical one; near-duplicates share the most frequent spelling."""
    unique = [name for name, _ in Counter(names).most_common()]  # most frequent first
    try:
        vectors = embed(unique)
    except Exception as e:  # no embedding model (e.g. Groq)? then merge nothing, but do not crash
        print("embedding failed, skipping merge:", e)
        return {name: name for name in unique}

    kept, mapping = [], {}
    for i, name in enumerate(unique):
        similarities = [(float(vectors[i] @ vectors[unique.index(k)]), k) for k in kept]
        best_score, best_name = max(similarities, default=(0.0, None))
        if best_score >= threshold:
            mapping[name] = best_name     # close enough to an existing aspect: merge into it
        else:
            mapping[name] = name          # new aspect
            kept.append(name)
    return mapping


# A look at the numbers behind the threshold.
pairs = [("noise cancelling", "noise cancellation"), ("usability", "ease of use"), ("price", "value"),
         ("noise cancelling", "anc"), ("sound quality", "call quality"), ("comfort", "price")]
vectors = embed([w for pair in pairs for w in pair])
for k, (a, b) in enumerate(pairs):
    print(f"{float(vectors[2 * k] @ vectors[2 * k + 1]):.2f}  {a} ~ {b}")

0.93  noise cancelling ~ noise cancellation
0.78  usability ~ ease of use
0.70  price ~ value
0.65  noise cancelling ~ anc
0.78  sound quality ~ call quality
0.58  comfort ~ price


Read those numbers before trusting the merge. Real duplicates score high (*noise cancelling* and
*noise cancellation*), but so does a pair that must **not** be merged (*sound quality* and *call quality*,
which share a word), while a real synonym scores low (*anc* for *noise cancelling*, an abbreviation the
embedding barely knows). Embeddings of one- or two-word labels carry little context, so the threshold is
set high (0.8) and the tool only merges near-identical phrasings. Exercise: embed each aspect together with
one of its quotes, or give the model a fixed list of aspects to choose from, and compare.

### 3c. Aggregate, and the complete tool

Now the full tool the agent will call: extract from every text, merge the names, count.

In [7]:
def analyze_reviews(texts: list[str], max_quotes: int = 2) -> list[dict]:
    """Aspect-based sentiment of a list of review texts: per aspect, how many positive/negative/mixed
    opinions were found and a few example quotes. Most-discussed aspects first."""
    opinions = []
    for text in texts:
        try:
            opinions += extract_opinions(text)
        except Exception as e:  # one bad page must not sink the whole analysis
            print("extraction failed for one text:", e)

    names = [o["aspect"].lower().strip() for o in opinions]
    canonical = merge_aspects(names)

    table: dict[str, dict] = {}
    for o, name in zip(opinions, names):
        aspect = canonical[name]
        row = table.setdefault(aspect, {"aspect": aspect, "positive": 0, "negative": 0, "mixed": 0, "quotes": []})
        row[o["sentiment"]] += 1
        if len(row["quotes"]) < max_quotes:
            row["quotes"].append(o["quote"])

    return sorted(table.values(), key=lambda r: -(r["positive"] + r["negative"] + r["mixed"]))


analysis = analyze_reviews(texts)
pd.DataFrame(analysis)

,aspect,positive,negative,mixed,quotes
0,sound quality,12,4,1,[The Sony WH-1000XM5 headphones offer decent s...
1,noise cancelling,13,0,0,"[It excels in travel or at the office, in part..."
2,comfort,8,1,0,[the padding of the Sony WH-1000XM5 is deep an...
3,battery life,3,0,1,[Our standard battery test yielded a result of...
4,build quality,0,4,0,[Build seems a little less premium than before...
5,value,2,2,0,[Slightly more expensive than closest rivals f...
6,design,2,1,0,"[With new drivers, features, and design, the S..."
7,call quality,3,0,0,"[great call quality, XM5 headphones deliver th..."
8,price,0,2,0,"[even at an eye-watering $400., More expensive..."
9,weight,2,0,0,[the WH-1000XM5 is light enough to spare your ...


### Reading the result

This is the "Customers say" table for the pages we fetched: which aspects people talk about most and
how the opinions split. The agent will turn it into a short, grounded summary; the quotes let it (and
you) check every claim against the source.

Two honest caveats:

* **Data, not algorithm, is the gap to Amazon.** We analysed three or four editorial reviews; Amazon
  aggregates thousands of customer reviews, so its counts mean something. With this little data the table
  is illustrative. The pipeline does not change with volume, only the tool that collects the texts.
* **Rate limits.** The tool makes one model call per text plus one embedding call. On the Gemini free tier
  that is fine for a handful of pages; a batch over thousands of reviews would need queueing and caching,
  which is why production runs it offline.

## 4. What makes a function a "tool"

Nothing in the code above is special. What turns these functions into tools is the contract they follow:

1. a descriptive **name** and a **docstring** that says what the function does and what it returns;
2. **typed arguments** with sensible defaults, so the model only has to supply the essentials;
3. a **JSON-serialisable return value** (strings, numbers, lists, dicts);
4. **no exceptions**: problems are reported as text;
5. **outputs sized for the prompt**: the result goes straight back into the conversation.

A tool may itself call a model, as `analyze_reviews` does, but with a fixed prompt and a fixed schema: it
is a function with a known input and output. In notebook 03 the name, docstring and argument types of
these functions become a JSON schema that is sent to the model alongside the messages, and the model
decides when to call which.